# Splitting Audio Samples into Chunks

by Tobias Erbacher

In this notebook, we take the lecture recordings (audio track only) and split them into 2-minute pieces with a rolling window of overlap set to 1 minute, i.e. the first sample will contain minute 0 - 2, the second sample will contain minute 1 - 3, etc. Note that this notebook was developed locally to save on computational costs and due to privacy considerations, for the latter reason, the audio files will not be uploaded to github. Moreover, we do not check whether the actual audio is spoken by the professor, or e.g. questions asked by the students, and therefore the model will inevitably not be completely accurate.

In [3]:
from pydub import AudioSegment
import os

INPUT_AUDIO_1 = "audio_files/converted_audio/V.1.1 - Variational Autoencoders.mp3"
INPUT_AUDIO_2 = "audio_files/converted_audio/V.2.1 - The Time Dimension- from Linear to Deep Predictors (first part).mp3"
INPUT_AUDIO_3 = "audio_files/converted_audio/V.6.1 - Leaning in Nonstationary Environments.mp3"

FILES_DICT = {
    INPUT_AUDIO_1 : "f1",
    INPUT_AUDIO_2 : "f2",
    INPUT_AUDIO_3 : "f3",
}

AUDIO_FILES = [INPUT_AUDIO_1, INPUT_AUDIO_2, INPUT_AUDIO_3]
AUDIO_OUTPUT_DIRECTORY = "audio_files/samples"

SAMPLE_LENGTH = 2 # Minutes
OVERLAP = 1 # Minutes

def split_mp3(input, output_directory, chunk_length, chunk_overlap):
    SAMPLE_DURATION = chunk_length * 60 * 1000
    SAMPLE_OVERLAP = chunk_overlap * 60 * 1000
    STEP_SIZE = SAMPLE_DURATION - SAMPLE_OVERLAP
    os.makedirs(output_directory, exist_ok=True)

    for file in AUDIO_FILES:
        audio = AudioSegment.from_mp3(file)
        audio_length = len(audio)

        c = 1
        for i in range(0, audio_length, STEP_SIZE):
            chunk = audio[i:i + SAMPLE_DURATION]
            chunk_name = os.path.join(output_directory, FILES_DICT[file] + "_sample_" + str(i // chunk_length + 1) + ".mp3")
            chunk.export(chunk_name, format="mp3")
            print("Saved chunk " + chunk_name)
            c += 1

split_mp3(AUDIO_FILES, AUDIO_OUTPUT_DIRECTORY, SAMPLE_LENGTH, OVERLAP)


Saved chunk audio_files/samples/f1_sample_1.mp3
Saved chunk audio_files/samples/f1_sample_30001.mp3
Saved chunk audio_files/samples/f1_sample_60001.mp3
Saved chunk audio_files/samples/f1_sample_90001.mp3
Saved chunk audio_files/samples/f1_sample_120001.mp3
Saved chunk audio_files/samples/f1_sample_150001.mp3
Saved chunk audio_files/samples/f1_sample_180001.mp3
Saved chunk audio_files/samples/f1_sample_210001.mp3
Saved chunk audio_files/samples/f1_sample_240001.mp3
Saved chunk audio_files/samples/f1_sample_270001.mp3
Saved chunk audio_files/samples/f1_sample_300001.mp3
Saved chunk audio_files/samples/f1_sample_330001.mp3
Saved chunk audio_files/samples/f1_sample_360001.mp3
Saved chunk audio_files/samples/f1_sample_390001.mp3
Saved chunk audio_files/samples/f1_sample_420001.mp3
Saved chunk audio_files/samples/f1_sample_450001.mp3
Saved chunk audio_files/samples/f1_sample_480001.mp3
Saved chunk audio_files/samples/f1_sample_510001.mp3
Saved chunk audio_files/samples/f1_sample_540001.mp3
S

Next, we infer the text spoken in each chunk with the openai-whisper library to generate the labels that we will later use to train the voice model.

In [4]:
import whisper

TRANSCRIPTION_PATH = AUDIO_OUTPUT_DIRECTORY

OPENAI_MODEL = whisper.load_model("base")

for file in os.listdir(TRANSCRIPTION_PATH):
    if file.endswith(".mp3"):
        file_path = os.path.join(TRANSCRIPTION_PATH, file)
        result = OPENAI_MODEL.transcribe(file_path)
        text = result["text"]

        text_file = os.path.splitext(file)[0] + ".txt"
        output_file_path = os.path.join(TRANSCRIPTION_PATH, text_file)

        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(text)
            print("Transcription saved to " + output_file_path)

/opt/anaconda3/lib/python3.11/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)
/opt/anac

Transcription saved to audio_files/samples/f2_sample_1800001.txt


/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to audio_files/samples/f3_sample_1470001.txt


/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


...

Transcription saved to audio_files/samples/f3_sample_1200001.txt


/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to audio_files/samples/f1_sample_2160001.txt


/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to audio_files/samples/f3_sample_2460001.txt
